# Case Centric Indexer
Similiar to gene-centric, but with case and gene reversed
```
case{}
     |___ gene[]
             |___ ssm[]
                   |___ consequence[]
                   |             |_____ transcript{}
                   |                          |_____ annotation{}
                   |___ observation[]
```

In [1]:
import os
import requests
import uuid
%load_ext autoreload
from exports.mappings import GeneMapper, SSMMapper, Mapper
from exports.utils import get_array_paths

from pyspark.sql.functions import col, explode, collect_list, size, sum, first, struct, udf, regexp_extract, lit, count, broadcast
from pyspark.sql.types import StringType

## Load combined maf into spark

In [2]:
url = 's3a://test/combined_mafs.csv'
    
df = sqlContext.read.format('com.databricks.spark.csv')\
                .options(header='true', inferschema='true')\
                .load(url)\
                .drop_duplicates()

## Rename and select desired columns in the mafs

In [5]:
%autoreload
from exports.utils import (
    maf_annotation_map,
    maf_gene_map,
    maf_observation_map,
    maf_ssm_map,
    maf_transcript_map,
    tumor_genotype_map,
    tumor_validation_map,
    normal_genotype_map,
    sample_map,
    input_bam_map,
    read_depth_map,
    maf_cols
)

maf_df = df.select(*( col(v).alias(k) for k,v in maf_cols.items() ))

## Augment maf df by extracting submitter_id and creating ssm_uuids

In [6]:
maf_df = maf_df.withColumn('_case_submitter_id',
                           regexp_extract(col('tumor_sample_barcode'),
                                          '([A-Z]{4}-[A-Z0-9]{2}-[A-Z0-9]{4})',1))
maf_ssm_map.update({'_case_submitter_id':'_case_submitter_id'})

In [7]:
def ssm_uuid(chromosome, start_position, ref_allele, tumor_allele):
    '''
    SNP: "{chromosome}:g.{start_position}{reference_allele}>{tumor_allele}"
    DEL: "{chromosome}:g.{start_position}del{reference_allele}"
    INS: "{chromosome}:g.{start_position}_{end_position}ins{tumor_allele}"
    '''
    chromosome = chromosome.replace('chr','')
    label = '{}:g.{}:{}>{}'.format(chromosome, start_position, ref_allele, tumor_allele)
    return str(uuid.uuid5(uuid.UUID('d15296a3-38ed-412e-8ace-75e235f82f55'), label))

ssm_uuid_udf =udf(ssm_uuid, StringType())
maf_df = maf_df.withColumn('ssm_id', ssm_uuid_udf(col('chromosome'), col('start_position'), col('reference_allele'), col('tumor_allele')))
maf_observation_map.update({'ssm_id':'ssm_id'})
maf_ssm_map.update({'ssm_id':'ssm_id'})

## Slice and dice until we get to the format we want

### Gene df

In [8]:
gene_df = maf_df.select(*( col(k) for k in maf_gene_map.keys() + ['_case_submitter_id'] ))
# Fill in empty data we don't know about
gene_df = gene_df.withColumn('description', lit(None).cast(StringType()))\
                 .drop_duplicates()

In [9]:
#gene_df.count()

### SSM df

In [10]:
ssm_df = maf_df.select(*( col(k) for k in maf_ssm_map.keys() ))\
               .drop_duplicates()

### Transcript-annotation df

```
transcript{}
     |_____ annotation{}
```

In [11]:
# Rename columns
tran_anno_df = maf_df.select('ssm_id',*( maf_transcript_map.keys() + maf_annotation_map.keys() ))
# Select annotation into nested format
tran_anno_df = tran_anno_df.select(struct(*maf_annotation_map.keys()).alias('annotation'), 'ssm_id', *maf_transcript_map.keys())\
                           .drop_duplicates()
#tran_anno_df.printSchema()

In [12]:
#tran_anno_df.count()

### Observation df

In [13]:
observation_df = maf_df.select(*(maf_observation_map.keys()
                                 +normal_genotype_map.keys()
                                 +tumor_genotype_map.keys()
                                 +tumor_validation_map.keys()
                                 +read_depth_map.keys()
                                 +input_bam_map.keys()
                                 +sample_map.keys()))

observation_df = observation_df.select('ssm_id',struct(*normal_genotype_map.keys()).alias('normal_genotype'),
                                       struct(*tumor_genotype_map.keys()).alias('tumor_genotype'),
                                       struct(*tumor_validation_map.keys()).alias('validation'),
                                       struct(*read_depth_map.keys()).alias('read_depth'),
                                       struct(*input_bam_map.keys()).alias('input_bam_file'),
                                       struct(*sample_map.keys()).alias('sample'),
                                       *maf_observation_map.keys())\
                                    .drop('gene_id')\
                                .drop_duplicates()

### Get case dataframe from existing graph

In [14]:
#doc = requests.get('http://elasticsearch.service.consul:9200/gdc_from_graph_35/_search').json()['hits']['hits'][0]['_source']
case_df = sqlContext.read.format("es")\
    .option('es.nodes', 'elasticsearch.service.consul')\
    .option('es.read.field.include', 'case_id,submitter_id,state,project.*,program.*')\
    .option('es.read.field.as.array.include','')\
    .option('es.resource.read', 'gdc_from_graph/case')\
    .option('es.nodes.resolve.hostname','false')\
    .load("gdc_from_graph")\
    .drop('state')
    
    #.option('es.read.field.include', 'case_id,submitter_id,state,project.*,program.*,exposures.*,demographic.*')\

## Assemble constituent parts

### Merge annotation with transcript

In [17]:
cons_tran_anno_df = tran_anno_df.select(struct(struct(*tran_anno_df.drop('gene_id').drop('ssm_id').columns).alias('transcript')).alias('consequence'),'ssm_id')\
                                .groupBy('ssm_id')\
                                .agg(collect_list('consequence').alias('consequence'))

In [18]:
#cons_tran_anno_df.count()

### Join observation with consequence

In [19]:
import random
def salt(key, doc_count=0):
    return str(random.randint(0,int(max(0,doc_count-1024)**5)))+key
salt_udf = udf(salt, StringType())

In [20]:
salted_observation_df = observation_df.withColumn('salt_key', salt_udf(col('ssm_id')))\
                                     #.repartition(1024, 'salt_key')
#salted_observation_df.persist().count()

In [21]:
salted_consequence_df = cons_tran_anno_df.withColumn('salt_key', salt_udf(col('ssm_id')))
                                         #.repartition(64, 'salt_key')
#salted_consequence_df.persist().count()

In [22]:
cons_obs = salted_consequence_df.join(salted_observation_df, salted_consequence_df.ssm_id == salted_observation_df.ssm_id, 'outer')\
                            .drop(salted_consequence_df.ssm_id)\
                            .drop(salted_consequence_df.salt_key)\
                            .select('ssm_id','consequence',struct(*salted_observation_df.drop('ssm_id').drop('salt_key').columns).alias('observation'))

### Join consequence into ssm

In [23]:
# Salt consequence-observation
salted_cons_obs = cons_obs.withColumn('doc_count', size(col('consequence')))\
                            .withColumn('salt_key', salt_udf(col('ssm_id'), col('doc_count')))

In [24]:
salted_ssm = ssm_df.withColumn('salt_key', salt_udf(col('gene_id')))

In [25]:
ssm_cons = salted_cons_obs.join(salted_ssm, salted_ssm.ssm_id == salted_cons_obs.ssm_id, 'left')\
                        .drop(salted_cons_obs.ssm_id)\
                        .drop(salted_cons_obs.salt_key)\
                        .select('gene_id', struct('consequence','observation',*ssm_df.drop('gene_id').drop('gene_id').drop('_case_submitter_id').columns).alias('ssm'))\
                        .groupBy('gene_id')\
                        .agg(collect_list('ssm').alias('ssm'))

### Join ssm with gene

In [26]:
gene_ssm = gene_df.join(ssm_cons, gene_df.gene_id == ssm_cons.gene_id, 'left')\
                    .drop(ssm_cons.gene_id)\
                    .select('_case_submitter_id', struct('ssm', *gene_df.drop('_case_submitter_id').columns).alias('gene'))

In [27]:
#ssm_cons.printSchema()

### Join case with gene

In [28]:
#gene_df.count()

In [29]:
case_centric = case_df.join(gene_ssm, case_df.submitter_id == gene_ssm._case_submitter_id, 'left')\
                        .groupBy(*case_df.columns)\
                        .agg(collect_list('gene').alias('gene'))

In [30]:
#case_centric.select(size('gene')).describe().show()

## Export df to es

In [31]:
sqlContext.sql("set spark.sql.shuffle.partitions=4096")

DataFrame[key: string, value: string]

#### Graph es index

In [32]:
#print requests.get('http://elasticsearch.service.consul:9200/_cat/indices?v').text

#### New vis index

In [33]:
index = 'dan-r2-case'

In [34]:
%autoreload
from exports.mappings import CaseMapper
m = CaseMapper()

In [37]:
import json

print requests.delete('http://elasticsearchvis.service.consul:9200/{}'.format(index)).json()

data = json.dumps({"settings":{"index":{
                "refresh_interval":"1m",
                "number_of_shards":20,
                "number_of_replicas":0,
                "mapper.dynamic":False,
                "mapping.nested_fields.limit":100,
                "mapping.total_fields.limit":2000
            }},"mappings":{
                "case":m.mapping
            }})
#print requests.put('http://localhost:9200/test/', data=data).json()
#print requests.put('http://elasticsearchvis.service.consul:9200/{}'.format(index), data=data).json()

ConnectionError: ('Connection aborted.', BadStatusLine("''",))

In [ ]:
#%%time
case_centric.coalesce(16).write.format('org.elasticsearch.spark.sql')\
                    .option('es.nodes', 'elasticsearchvis.service.consul')\
                    .option('es.nodes.resolve.hostname','false')\
                    .option('es.resource.write', '{}/case'.format(index))\
                    .option('es.http.timeout', '1m')\
                    .option('es.http.retries', '300')\
                    .option('es.batch.write.retry.count','-1')\
                    .option('es.batch.write.retry.wait', '100s')\
                    .option('es.batch.size.bytes','100mb')\
                    .option('es.batch.size.entries', '1')\
                    .option('es.mapping.id','case_id')\
                    .save('{}/case'.format(index))

In [ ]:
requests.post('http://elasticsearchvis.service.consul:9200/{}/_refresh'.format(index))

In [ ]:
#requests.get('http://localhost:9200/test/_search?size=5').json()['hits']['hits']

In [ ]:
test_query = {
  "query": {
    "nested": {
      "path": "case",
      "score_mode": "sum",
      "query": {
        "function_score": {
          "query": {
            "bool": {
              "must": [
                {
                "terms": {
                  "case.project.project_id": [
                    "TCGA-ACC"
                  ]
                }
                }
              ]
            }
          }
        }
      }
    }
  }
}
      
len(requests.post('http://localhost:9200/test/_search', data=json.dumps(test_query)).json()['hits']['hits'])